# Team A Solution — Molecular Spectrum Estimation with Quantum Phase Estimation

We solve the workshop exactly in the reduced, self-contained setting:

1. write down the three effective two-qubit molecular Hamiltonians,
2. diagonalize them classically,
3. build a QPE circuit for $\mathrm{H}_2$,
4. recover an energy estimate from the dominant measured phase,
5. repeat the same logic for $\mathrm{LiH}$ and $\mathrm{BeH}_2$.

The central phase-to-energy relation is

$$
U' = e^{-i(H-sI)\tau},
\qquad
E = s - \frac{2\pi}{\tau}\phi.
$$

In this notebook, every major algebraic step is followed by the exact code used to implement it.

In [ ]:
import os
import warnings
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mpl-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(Path.cwd() / ".cache"))
Path(os.environ["XDG_CACHE_HOME"]).mkdir(exist_ok=True)
os.environ.setdefault("MPLBACKEND", "Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.linalg import expm
from IPython.display import Markdown, display
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
from qiskit.circuit.library import QFTGate
from qiskit.circuit.library.generalized_gates import UnitaryGate

warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive, and thus cannot be shown")

np.set_printoptions(precision=5, suppress=True)
pd.options.display.float_format = "{:.6f}".format
plt.style.use("tableau-colorblind10")

## Step 1 — Define the Pauli Matrices and the Molecular Coefficients

Each molecule is modeled by

$$
H = c_{II}II + c_{ZI}ZI + c_{IZ}IZ + c_{ZZ}ZZ + c_{XX}XX.
$$

We first record the coefficients exactly as given in the workshop statement.

In [ ]:
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

h2_coeffs = {"II": -1.05, "ZI": 0.39, "IZ": 0.39, "ZZ": -0.01, "XX": 0.18}
lih_coeffs = {"II": -2.20, "ZI": 0.18, "IZ": 0.12, "ZZ": -0.35, "XX": 0.28}
beh2_coeffs = {"II": -3.60, "ZI": 0.25, "IZ": 0.31, "ZZ": -0.48, "XX": 0.41}

coefficient_table = pd.DataFrame(
    [
        ["H2", *h2_coeffs.values()],
        ["LiH", *lih_coeffs.values()],
        ["BeH2", *beh2_coeffs.values()],
    ],
    columns=["Molecule", "II", "ZI", "IZ", "ZZ", "XX"],
)
display(coefficient_table)

## Step 2 — Build the Three Hamiltonian Matrices Explicitly

Now we translate the coefficient dictionaries into actual $4\times 4$ matrices.

This is the point where the abstract Pauli expansion becomes a concrete matrix eigenvalue problem.

In [ ]:
H_h2 = (
    h2_coeffs["II"] * np.kron(I, I)
    + h2_coeffs["ZI"] * np.kron(Z, I)
    + h2_coeffs["IZ"] * np.kron(I, Z)
    + h2_coeffs["ZZ"] * np.kron(Z, Z)
    + h2_coeffs["XX"] * np.kron(X, X)
)

H_lih = (
    lih_coeffs["II"] * np.kron(I, I)
    + lih_coeffs["ZI"] * np.kron(Z, I)
    + lih_coeffs["IZ"] * np.kron(I, Z)
    + lih_coeffs["ZZ"] * np.kron(Z, Z)
    + lih_coeffs["XX"] * np.kron(X, X)
)

H_beh2 = (
    beh2_coeffs["II"] * np.kron(I, I)
    + beh2_coeffs["ZI"] * np.kron(Z, I)
    + beh2_coeffs["IZ"] * np.kron(I, Z)
    + beh2_coeffs["ZZ"] * np.kron(Z, Z)
    + beh2_coeffs["XX"] * np.kron(X, X)
)

print("H2 Hamiltonian:")
print(H_h2)
print("\nLiH Hamiltonian:")
print(H_lih)
print("\nBeH2 Hamiltonian:")
print(H_beh2)

## Step 3 — Diagonalize the Hamiltonians Classically

Since QPE is an eigenvalue algorithm, the exact classical diagonalization is our benchmark.

We extract:

- the four exact energies for each molecule,
- the ground-to-first-excited gap,
- the overlap of the simple reference state $|11\rangle$ with the exact ground state.

In [ ]:
h2_evals, h2_evecs = np.linalg.eigh(H_h2)
lih_evals, lih_evecs = np.linalg.eigh(H_lih)
beh2_evals, beh2_evecs = np.linalg.eigh(H_beh2)

ket_11 = np.array([0, 0, 0, 1], dtype=complex)
h2_overlaps = np.abs(h2_evecs.conj().T @ ket_11) ** 2
lih_overlaps = np.abs(lih_evecs.conj().T @ ket_11) ** 2
beh2_overlaps = np.abs(beh2_evecs.conj().T @ ket_11) ** 2

spectrum_table = pd.DataFrame(
    [
        ["H2", h2_evals[0], h2_evals[1], h2_evals[1] - h2_evals[0], h2_overlaps[0]],
        ["LiH", lih_evals[0], lih_evals[1], lih_evals[1] - lih_evals[0], lih_overlaps[0]],
        ["BeH2", beh2_evals[0], beh2_evals[1], beh2_evals[1] - beh2_evals[0], beh2_overlaps[0]],
    ],
    columns=["Molecule", "Ground energy", "1st excited", "Gap E1-E0", "Ground overlap of |11>"],
)
display(spectrum_table)

level_table = pd.DataFrame(
    {
        "Level": ["E0", "E1", "E2", "E3"],
        "H2": h2_evals,
        "LiH": lih_evals,
        "BeH2": beh2_evals,
    }
)
display(level_table)

## Step 4 — Visualize the Exact Spectra

The left panel compares the exact energy levels across the three effective molecular models.

The right panel shows why $|11\rangle$ is a good teaching-state choice: it already has strong ground-state overlap in all three models.

In [ ]:
molecule_positions = np.arange(3)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for level_index, level_name, color in zip(
    range(4),
    ["E0", "E1", "E2", "E3"],
    ["#1d4ed8", "#ea580c", "#0f766e", "#7c3aed"],
):
    axes[0].plot(
        molecule_positions,
        [h2_evals[level_index], lih_evals[level_index], beh2_evals[level_index]],
        marker="o",
        linewidth=2,
        label=level_name,
        color=color,
    )

axes[0].set_xticks(molecule_positions, ["H2", "LiH", "BeH2"])
axes[0].set_ylabel("Energy")
axes[0].set_title("Exact energy spectra", loc="left")
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].bar(
    ["H2", "LiH", "BeH2"],
    [h2_overlaps[0], lih_overlaps[0], beh2_overlaps[0]],
    color=["#2563eb", "#ea580c", "#0f766e"],
)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel(r"$|\langle E_0 | 11 \rangle|^2$")
axes[1].set_title("Ground-state support of |11>", loc="left")
axes[1].grid(axis="y", alpha=0.25)

fig.tight_layout()
display(fig)
plt.close(fig)

## Step 5 — Shift the Hamiltonian for QPE

QPE returns a phase, not an energy directly.

To keep the phase map clean, we shift the Hamiltonian by a scalar:

$$
H' = H - sI,
\qquad
s = E_{\max} + 0.10.
$$

We use $\tau = 0.70$ and inspect the exact phases that should be seen for $\mathrm{H}_2$.

In [ ]:
tau = 0.70
phase_qubits = 6

h2_shift = float(h2_evals.max() + 0.10)
H_h2_shifted = H_h2 - h2_shift * np.eye(4)
U_h2 = expm(-1j * H_h2_shifted * tau)
h2_shifted_phases = -(h2_evals - h2_shift) * tau / (2 * np.pi)

expected_phase_table = pd.DataFrame(
    {
        "Level": ["E0", "E1", "E2", "E3"],
        "Exact energy": h2_evals,
        "Expected phase": h2_shifted_phases,
    }
)
display(expected_phase_table)

## Step 5B — Build the Same $\mathrm{H}_2$ Evolution from Native Gates

The Hamiltonian itself is not a gate, but its time evolution is.

For

$$
H = c_{II}II + c_{ZI}ZI + c_{IZ}IZ + c_{ZZ}ZZ + c_{XX}XX,
$$

the identity term is only a global phase, so the nontrivial evolution can be approximated directly with:

- `rz` for the single-qubit $Z$ terms,
- `rzz` for the $ZZ$ term,
- a basis change $H \otimes H$, then `rzz`, then $H \otimes H$ for the $XX$ term.

Because the $XX$ term does not commute with the $Z$ terms, this is a first-order Trotter step rather than the exact matrix exponential.

In [ ]:
h2_gate_evolution = QuantumCircuit(2)

h2_gate_evolution.rz(2 * tau * h2_coeffs["ZI"], 0)
h2_gate_evolution.rz(2 * tau * h2_coeffs["IZ"], 1)
h2_gate_evolution.rzz(2 * tau * h2_coeffs["ZZ"], 0, 1)

h2_gate_evolution.h(0)
h2_gate_evolution.h(1)
h2_gate_evolution.rzz(2 * tau * h2_coeffs["XX"], 0, 1)
h2_gate_evolution.h(0)
h2_gate_evolution.h(1)

trotter_matrix = Operator(h2_gate_evolution).data
phase_alignment = np.angle(np.vdot(trotter_matrix.reshape(-1), U_h2.reshape(-1)))
trotter_matrix_aligned = np.exp(1j * phase_alignment) * trotter_matrix
trotter_error = np.linalg.norm(U_h2 - trotter_matrix_aligned)

print(h2_gate_evolution.draw(output="text"))

trotter_table = pd.DataFrame(
    [["|| U_exact - e^{iα} U_trotter ||", trotter_error]],
    columns=["Check", "Value"],
)
display(trotter_table)

## Step 6 — Build the Exact-Eigenstate QPE Circuit for $\mathrm{H}_2$

This is the clean reference case:

- the phase register starts in a uniform superposition,
- the system register starts in the exact ground-state eigenvector,
- controlled powers of $U' = e^{-i(H-sI)\tau}$ are applied,
- inverse QFT decodes the phase into a bitstring.

In [ ]:
h2_exact_qpe = QuantumCircuit(phase_qubits + 2)
for q in range(phase_qubits):
    h2_exact_qpe.h(q)

h2_exact_qpe.initialize(h2_evecs[:, 0], [phase_qubits, phase_qubits + 1])

for q in range(phase_qubits):
    powered_gate = UnitaryGate(np.linalg.matrix_power(U_h2, 2**q), label=f"U^{2**q}")
    h2_exact_qpe.append(powered_gate.control(1), [q, phase_qubits, phase_qubits + 1])

h2_exact_qpe.append(QFTGate(phase_qubits).inverse(), range(phase_qubits))

print(h2_exact_qpe.draw(output="text"))

## Step 7 — Run Exact-Eigenstate QPE and Decode the Energy

If the input is an exact eigenstate, the histogram should collapse sharply onto one dominant phase bitstring.

In [ ]:
h2_exact_statevector = Statevector.from_instruction(h2_exact_qpe)
h2_exact_distribution = h2_exact_statevector.probabilities_dict(qargs=list(range(phase_qubits)))

h2_exact_top = sorted(
    [(str(bitstring), float(probability)) for bitstring, probability in h2_exact_distribution.items()],
    key=lambda item: item[1],
    reverse=True,
)[:8]

h2_exact_rows = []
for bitstring, probability in h2_exact_top:
    phase = int(bitstring, 2) / (2**phase_qubits)
    energy = h2_shift - (2 * np.pi / tau) * phase
    h2_exact_rows.append([bitstring, phase, probability, energy])

h2_exact_table = pd.DataFrame(
    h2_exact_rows,
    columns=["Bitstring", "Phase", "Probability", "Recovered energy"],
)
display(h2_exact_table)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.bar(h2_exact_table["Bitstring"], h2_exact_table["Probability"], color="#2563eb")
ax.set_title("H2 QPE histogram from exact ground-state initialization", loc="left")
ax.set_xlabel("Measured phase bitstring")
ax.set_ylabel("Probability")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45)
plt.tight_layout()
display(fig)
plt.close(fig)

dominant_bitstring = h2_exact_table.loc[0, "Bitstring"]
dominant_phase = h2_exact_table.loc[0, "Phase"]
dominant_energy = h2_exact_table.loc[0, "Recovered energy"]
dominant_probability = h2_exact_table.loc[0, "Probability"]

display(Markdown(
    rf'''
    Dominant result:

    - bitstring: `{dominant_bitstring}`
    - phase: `{dominant_phase:.6f}`
    - probability: `{dominant_probability:.6f}`
    - recovered energy: `{dominant_energy:.6f}`
    - exact ground energy: `{h2_evals[0]:.6f}`
    - absolute error: `{abs(dominant_energy - h2_evals[0]):.6f}`
    '''
))

## Step 8 — Build the Reference-State QPE Circuit from $|11\rangle$

This is the more realistic teaching example: we no longer prepare an exact eigenvector.

Instead, we begin with the simple computational basis state $|11\rangle$, so the QPE histogram reflects the decomposition

$$
|11\rangle = \sum_j c_j |E_j\rangle.
$$

In [ ]:
h2_reference_qpe = QuantumCircuit(phase_qubits + 2)
for q in range(phase_qubits):
    h2_reference_qpe.h(q)

h2_reference_qpe.x(phase_qubits)
h2_reference_qpe.x(phase_qubits + 1)

for q in range(phase_qubits):
    powered_gate = UnitaryGate(np.linalg.matrix_power(U_h2, 2**q), label=f"U^{2**q}")
    h2_reference_qpe.append(powered_gate.control(1), [q, phase_qubits, phase_qubits + 1])

h2_reference_qpe.append(QFTGate(phase_qubits).inverse(), range(phase_qubits))

print(h2_reference_qpe.draw(output="text"))

## Step 9 — Run Reference-State QPE and Compare with Exact Overlaps

Now the phase histogram should reveal the eigenstate weights of the reference state.

We compare the observed QPE distribution with the exact overlap values $|\langle E_j|11\rangle|^2$.

In [ ]:
h2_reference_statevector = Statevector.from_instruction(h2_reference_qpe)
h2_reference_distribution = h2_reference_statevector.probabilities_dict(qargs=list(range(phase_qubits)))

h2_reference_top = sorted(
    [(str(bitstring), float(probability)) for bitstring, probability in h2_reference_distribution.items()],
    key=lambda item: item[1],
    reverse=True,
)[:8]

h2_reference_rows = []
for bitstring, probability in h2_reference_top:
    phase = int(bitstring, 2) / (2**phase_qubits)
    energy = h2_shift - (2 * np.pi / tau) * phase
    h2_reference_rows.append([bitstring, phase, probability, energy])

h2_reference_table = pd.DataFrame(
    h2_reference_rows,
    columns=["Bitstring", "Phase", "Probability", "Recovered energy"],
)
display(h2_reference_table)

overlap_table = pd.DataFrame(
    {
        "Level": ["E0", "E1", "E2", "E3"],
        "Exact energy": h2_evals,
        "Overlap with |11>": h2_overlaps,
    }
)
display(overlap_table)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

axes[0].bar(h2_reference_table["Bitstring"], h2_reference_table["Probability"], color="#ea580c")
axes[0].set_title("H2 QPE histogram from |11> initialization", loc="left")
axes[0].set_xlabel("Measured phase bitstring")
axes[0].set_ylabel("Probability")
axes[0].grid(axis="y", alpha=0.25)
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(overlap_table["Level"], overlap_table["Overlap with |11>"], color="#0f766e")
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Exact overlap of |11> with the H2 eigenbasis", loc="left")
axes[1].set_ylabel("Weight")
axes[1].grid(axis="y", alpha=0.25)

fig.tight_layout()
display(fig)
plt.close(fig)

## Step 10 — Repeat Ground-State QPE for $\mathrm{LiH}$ and $\mathrm{BeH}_2$

We now run the same exact-eigenstate QPE logic for all three molecules and compare the recovered ground-state energies.

We also track how the absolute error changes as the number of phase qubits increases from 4 to 7.

In [ ]:
molecule_names = ["H2", "LiH", "BeH2"]
hamiltonians = [H_h2, H_lih, H_beh2]
eigenvalues = [h2_evals, lih_evals, beh2_evals]
eigenvectors = [h2_evecs, lih_evecs, beh2_evecs]
overlaps = [h2_overlaps, lih_overlaps, beh2_overlaps]

summary_rows = []
error_rows = []

for molecule_name, H_matrix, evals, evecs, ref_overlaps in zip(
    molecule_names, hamiltonians, eigenvalues, eigenvectors, overlaps
):
    shift = float(evals.max() + 0.10)
    U_matrix = expm(-1j * (H_matrix - shift * np.eye(4)) * tau)

    qpe_circuit = QuantumCircuit(phase_qubits + 2)
    for q in range(phase_qubits):
        qpe_circuit.h(q)
    qpe_circuit.initialize(evecs[:, 0], [phase_qubits, phase_qubits + 1])
    for q in range(phase_qubits):
        powered_gate = UnitaryGate(np.linalg.matrix_power(U_matrix, 2**q), label=f"U^{2**q}")
        qpe_circuit.append(powered_gate.control(1), [q, phase_qubits, phase_qubits + 1])
    qpe_circuit.append(QFTGate(phase_qubits).inverse(), range(phase_qubits))

    distribution = Statevector.from_instruction(qpe_circuit).probabilities_dict(qargs=list(range(phase_qubits)))
    dominant_bitstring, dominant_probability = max(
        ((str(bitstring), float(probability)) for bitstring, probability in distribution.items()),
        key=lambda item: item[1],
    )
    dominant_phase = int(dominant_bitstring, 2) / (2**phase_qubits)
    recovered_energy = shift - (2 * np.pi / tau) * dominant_phase

    summary_rows.append(
        [
            molecule_name,
            evals[0],
            dominant_bitstring,
            dominant_phase,
            recovered_energy,
            abs(recovered_energy - evals[0]),
            ref_overlaps[0],
        ]
    )

    row = {"Molecule": molecule_name}
    for m in [4, 5, 6, 7]:
        qpe_circuit = QuantumCircuit(m + 2)
        for q in range(m):
            qpe_circuit.h(q)
        qpe_circuit.initialize(evecs[:, 0], [m, m + 1])
        for q in range(m):
            powered_gate = UnitaryGate(np.linalg.matrix_power(U_matrix, 2**q), label=f"U^{2**q}")
            qpe_circuit.append(powered_gate.control(1), [q, m, m + 1])
        qpe_circuit.append(QFTGate(m).inverse(), range(m))

        distribution = Statevector.from_instruction(qpe_circuit).probabilities_dict(qargs=list(range(m)))
        dominant_bitstring, dominant_probability = max(
            ((str(bitstring), float(probability)) for bitstring, probability in distribution.items()),
            key=lambda item: item[1],
        )
        dominant_phase = int(dominant_bitstring, 2) / (2**m)
        recovered_energy = shift - (2 * np.pi / tau) * dominant_phase
        row[f"{m} qubits"] = abs(recovered_energy - evals[0])

    error_rows.append(row)

summary_table = pd.DataFrame(
    summary_rows,
    columns=[
        "Molecule",
        "Exact ground energy",
        "Dominant bitstring",
        "Phase",
        "Recovered energy",
        "Absolute error",
        "Ground overlap of |11>",
    ],
)
display(summary_table)

error_table = pd.DataFrame(error_rows)
display(error_table)

## Step 11 — Visual Comparison Across the Three Molecules

The first panel compares the exact and recovered ground-state energies.

The second panel shows how the QPE error changes with phase-register size.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

x = np.arange(len(summary_table))
width = 0.36

axes[0].bar(x - width / 2, summary_table["Exact ground energy"], width=width, label="Exact", color="#334155")
axes[0].bar(x + width / 2, summary_table["Recovered energy"], width=width, label="QPE", color="#2563eb")
axes[0].set_xticks(x, summary_table["Molecule"])
axes[0].set_title("Exact versus recovered ground-state energies", loc="left")
axes[0].set_ylabel("Energy")
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend()

phase_sizes = [4, 5, 6, 7]
for molecule_name, color in zip(["H2", "LiH", "BeH2"], ["#2563eb", "#ea580c", "#0f766e"]):
    row = error_table.loc[error_table["Molecule"] == molecule_name].iloc[0]
    axes[1].plot(
        phase_sizes,
        [row["4 qubits"], row["5 qubits"], row["6 qubits"], row["7 qubits"]],
        marker="o",
        linewidth=2,
        label=molecule_name,
        color=color,
    )

axes[1].set_xticks(phase_sizes)
axes[1].set_xlabel("Number of phase qubits")
axes[1].set_ylabel("Absolute ground-state error")
axes[1].set_title("QPE error versus phase-register size", loc="left")
axes[1].grid(alpha=0.25)
axes[1].legend()

fig.tight_layout()
display(fig)
plt.close(fig)

## Step 12 — Final Interpretation

The workshop logic is now complete:

- the molecular models are explicit,
- the exact spectra are known,
- QPE is implemented directly on the shifted time-evolution operator,
- the recovered phase is converted back into an energy estimate,
- the behavior across $\mathrm{H}_2$, $\mathrm{LiH}$, and $\mathrm{BeH}_2$ can be compared quantitatively.

The main scientific lesson is unchanged:

- QPE genuinely estimates eigenvalues through phase,
- classical diagonalization is the correctness benchmark,
- these reduced two-qubit models are pedagogically useful,
- realistic chemistry remains hard because of state preparation and Hamiltonian simulation cost.